In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datetime import datetime

In [3]:
curve_df = pd.read_hdf('/media/ssd1/huong/PCR-huong/data/new_data.h5', key='curve_data')
sample_info = pd.read_hdf('/media/ssd1/huong/PCR-huong/data/new_data.h5', key='sample_info')
sample_info = (sample_info
               .loc[~sample_info.sample_id.isin(['S1268904','S1268905'])])
igi_gene_call = pd.read_hdf('/media/ssd1/huong/PCR-huong/data/new_data.h5', key='igi_gene_call')

join_df = (curve_df
            .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
            .merge(igi_gene_call, how='inner', on=['pcr_plate','sample_id','target']))

join_df['created_date'] = join_df.created_date.apply(lambda x: datetime.strptime(x,'%m/%d/%y')) 
print(join_df.shape)           

(33032145, 26)


In [4]:
join_df.head()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,...,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
0,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,1,13408.349609,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
1,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,2,13445.377930,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
2,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,3,13458.586914,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
3,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,4,13539.398438,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
4,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,5,13588.847656,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367


In [4]:
join_df.columns

Index(['well_position', 'target', 'dye', 'amp_score', 'cq', 'threshold',
       'baseline_start', 'baseline_end', 'cycle_no', 'rn', 'drn', 'Fn',
       'pcr_plate', 'curve_idx', 'sample_id', 'sample_barcode', 'sample_type',
       'final_patient_result', 'current_sample_result', 'created_date',
       'record_type', 'retest_sample_id_1', 'retest_sample_id_2', 'file',
       'igi_call', 'thres_ct'],
      dtype='object')

In [4]:
clinical_ctrl_genes = (join_df[(join_df.retest_sample_id_1.isna())]
                       .loc[(join_df.sample_type == 'Clinical Sample') & (join_df.target.isin(['MS2','RnaseP'])), 
                            ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                       .copy())
clinical_ctrl_genes.loc[:,'groundtruth'] = 1

In [5]:
pos_ctrl_sample = (join_df
                   .loc[(join_df.sample_type == 'Positive Control (qPCR)'),
                        ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                   .copy())
pos_ctrl_sample.loc[:,'groundtruth'] = 1
pos_ctrl_sample.loc[pos_ctrl_sample.target.isin(['MS2','RnaseP']),'groundtruth'] = -1

In [6]:
(pos_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene   1             3123
MS2     -1              809
N gene   1             3932
ORF1ab   1              809
RnaseP  -1             3123
S gene   1              809
Name: curve_idx, dtype: int64

In [7]:
human_ctrl_sample = (join_df
                     .loc[(join_df.sample_type == 'Human Normal Negative Control (Extraction)'),
                          ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                     .copy())
human_ctrl_sample.loc[:,'groundtruth'] = -1
human_ctrl_sample.loc[human_ctrl_sample.target.isin(['MS2','RnaseP']),'groundtruth'] = 1

In [8]:
(human_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene  -1             1963
MS2      1              441
N gene  -1             2404
ORF1ab  -1              441
RnaseP   1             1963
S gene  -1              441
Name: curve_idx, dtype: int64

In [9]:
neg_ctrl_sample = (join_df
                   .loc[(join_df.sample_type == 'Negative Control (qPCR)'),
                        ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                   .copy())
neg_ctrl_sample.loc[:,'groundtruth'] = -1

In [10]:
(neg_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene  -1             3394
MS2     -1              809
N gene  -1             4203
ORF1ab  -1              809
RnaseP  -1             3394
S gene  -1              809
Name: curve_idx, dtype: int64

In [37]:
# buffer_ctrl_sample = (join_df
#                       .loc[(join_df.sample_type == 'Buffer Negative Control (Extraction)') & (~join_df.target.isin(['MS2','RnaseP'])),
#                            ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
#                       .copy())
# buffer_ctrl_sample.loc[:,'groundtruth'] = -1

In [38]:
# (buffer_ctrl_sample
#  .groupby(['target','groundtruth'])
#  .curve_idx.nunique())

In [11]:
groundtruth_df = pd.concat([neg_ctrl_sample, pos_ctrl_sample, 
                            human_ctrl_sample, clinical_ctrl_genes])

In [12]:
final_patient_df = (groundtruth_df[['sample_id','sample_type']]
 .drop_duplicates()
 .merge(join_df[['sample_id','final_patient_result']].drop_duplicates()))
 

In [13]:
join_df.loc[(join_df.sample_type == 'Clinical Sample') & (join_df.final_patient_result.isna()),'pcr_plate'].drop_duplicates()

1617965     AC00DBTC
2560220     C302NBGZ
3469920     C302NBGA
5314325     AC00DAV7
6480485     C302NBSC
6804305     C302JHUR
8248360     AC00DAPD
8645490     AC00DAS2
8650585     AC00DAZC
10833805    C302JHWC
11009720    C302NBWK
12249305    C302NBW3
12771185    AC00H0UT
13454655    AC00DBSV
13826920    C302NBCZ
14647665    C302NBE4
15255965    AC00DBSW
15420110    C302NBLV
15604635    C302JKVG
17174555    C302NBGC
18221875    C302NBG3
18613740    C302JKXW
19022850    AC00DBTA
19349460    C302NBG2
19791495    AC00GY11
20878520    AC00DAP8
21384070    C302JHUC
23624740    AC00DAPI
23669660    C302JHUK
24060850    AC00DAZU
24358320    AC00DAPG
24498465    AC00DAUV
25583550    AC00DC43
26810225    AC00DC4C
27839525    C302ST2B
28102560    C302ST2E
30136050    AC00DBSU
31254000    AC00DB0T
32086410    C302NBF0
32673330    C302NBEA
Name: pcr_plate, dtype: object

In [14]:
join_df.loc[(join_df.sample_type == 'Clinical Sample') & (join_df.final_patient_result.isna()),['sample_id','sample_barcode', 'pcr_plate']].drop_duplicates()

,sample_id,sample_barcode,pcr_plate
1617965,S134853,LLIGI0000040293- Retest 1,AC00DBTC
1618605,S134925,LLIGI0000037840- Retest 1,AC00DBTC
1618925,S134920,LLIGI0000043587- Retest 1,AC00DBTC
1619245,S134919,LLIGI0000045136- Retest 1,AC00DBTC
1619565,S134907,UHSS0000006314- Retest 1,AC00DBTC
...,...,...,...
32144650,S121470,UHSS0000004775- Retest 1,C302NBF0
32144970,S121403,UHSS0000004766- Retest 1,C302NBF0
32145290,S121498,UHSS0000005751- Retest 1,C302NBF0
32673330,S105415,PCIGI0000000312- Retest 1,C302NBEA


In [17]:
check_df = final_patient_df[(final_patient_df.sample_type == 'Clinical Sample') & (final_patient_df.final_patient_result.isna())]
join_df[join_df.sample_id.isin(check_df.sample_id)]

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,...,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
1617965,A2,S gene,ABY,1.777012,UNDETERMINED,41725.750494,3,39,1,187224.468750,...,Clinical Sample,NaN,Negative,2020-08-26,Re-Test Sample,NaN,NaN,dbtc,Negative,Undetermined
1617966,A2,S gene,ABY,1.777012,UNDETERMINED,41725.750494,3,39,2,187135.187500,...,Clinical Sample,NaN,Negative,2020-08-26,Re-Test Sample,NaN,NaN,dbtc,Negative,Undetermined
1617967,A2,S gene,ABY,1.777012,UNDETERMINED,41725.750494,3,39,3,186842.734375,...,Clinical Sample,NaN,Negative,2020-08-26,Re-Test Sample,NaN,NaN,dbtc,Negative,Undetermined
1617968,A2,S gene,ABY,1.777012,UNDETERMINED,41725.750494,3,39,4,186094.234375,...,Clinical Sample,NaN,Negative,2020-08-26,Re-Test Sample,NaN,NaN,dbtc,Negative,Undetermined
1617969,A2,S gene,ABY,1.777012,UNDETERMINED,41725.750494,3,39,5,185703.078125,...,Clinical Sample,NaN,Negative,2020-08-26,Re-Test Sample,NaN,NaN,dbtc,Negative,Undetermined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32699405,K6,MS2,JUN,2.581035,24.09136162132279,7143.000000,3,19,36,653448.812500,...,Clinical Sample,NaN,Invalid,2020-08-04,Re-Test Sample,NaN,NaN,nbea,Positive,24.09136162
32699406,K6,MS2,JUN,2.581035,24.09136162132279,7143.000000,3,19,37,682946.875000,...,Clinical Sample,NaN,Invalid,2020-08-04,Re-Test Sample,NaN,NaN,nbea,Positive,24.09136162
32699407,K6,MS2,JUN,2.581035,24.09136162132279,7143.000000,3,19,38,711588.375000,...,Clinical Sample,NaN,Invalid,2020-08-04,Re-Test Sample,NaN,NaN,nbea,Positive,24.09136162
32699408,K6,MS2,JUN,2.581035,24.09136162132279,7143.000000,3,19,39,738098.437500,...,Clinical Sample,NaN,Invalid,2020-08-04,Re-Test Sample,NaN,NaN,nbea,Positive,24.09136162


In [19]:
final_patient_df.loc[(final_patient_df.sample_type == 'Negative Control (qPCR)') & (final_patient_df.final_patient_result.isna()), 'final_patient_result'] = 'Negative'
final_patient_df.loc[(final_patient_df.sample_type == 'Positive Control (qPCR)') & (final_patient_df.final_patient_result.isna()), 'final_patient_result'] = 'Positive'
final_patient_df.loc[(final_patient_df.sample_type == 'Human Normal Negative Control (Extraction)') & (final_patient_df.final_patient_result.isna()), 'final_patient_result'] = 'Negative'
final_patient_df.loc[(final_patient_df.sample_type == 'Clinical Sample') & (final_patient_df.final_patient_result.isna()), 'final_patient_result'] = 'Retest'

In [20]:
data_ids = final_patient_df.sample_id
print('Number of sample_id in groundtruth:', len(data_ids))
labels = [1]*len(data_ids)
stratified_col = final_patient_df.final_patient_result


Number of sample_id in groundtruth: 218565


In [33]:

# stratified_col = [groundtruth_df.target[i] + '_' + groundtruth_df.groundtruth[i] for i in range(len()) ]
train_ids, test_ids, train_labels, test_labels = train_test_split(data_ids, stratified_col, test_size=0.1, stratify=stratified_col, random_state=1)
train_ids, val_ids, train_labels, val_labels = train_test_split(train_ids, train_labels, test_size=0.11, stratify=final_patient_df[final_patient_df.sample_id.isin(train_ids)].final_patient_result, random_state=1)

In [34]:
groundtruth_df.loc[:,'split'] = 'train'
groundtruth_df.loc[groundtruth_df.sample_id.isin(test_ids), 'split'] = 'test'
groundtruth_df.loc[groundtruth_df.sample_id.isin(val_ids), 'split'] = 'val'


In [35]:
groundtruth_df.groupby(['split']).sample_id.nunique()

split
test      21857
train    175070
val       21638
Name: sample_id, dtype: int64

In [37]:
groundtruth_df.groupby(['split']).curve_idx.nunique()

split
test      24175
train    193558
val       23961
Name: curve_idx, dtype: int64

In [38]:
(groundtruth_df
 .groupby(['split','groundtruth'])
 .curve_idx.nunique())

split  groundtruth
test   -1               2325
        1              21850
train  -1              18047
        1             175511
val    -1               2242
        1              21719
Name: curve_idx, dtype: int64

In [40]:
groundtruth_df.to_csv('/media/ssd1/huong/PCR-huong/data/new_groundtruth_df.csv', index = False)

In [2]:
groundtruth = pd.read_csv('/media/ssd1/huong/PCR-huong/data/new_groundtruth_df.csv')

In [6]:
groundtruth.groupby(['split']).curve_idx.nunique().reset_index()

,split,curve_idx
0,test,7252
1,train,175842
2,val,58627


In [7]:
groundtruth.groupby(['groundtruth']).curve_idx.nunique().reset_index()

,groundtruth,curve_idx
0,-1,22599
1,1,219077


In [8]:
groundtruth.groupby(['split', 'groundtruth']).curve_idx.nunique().reset_index()

,split,groundtruth,curve_idx
0,test,-1,702
1,test,1,6550
2,train,-1,16473
3,train,1,159369
4,val,-1,5452
5,val,1,53175


In [31]:
groundtruth.groupby(['test_kit', 'target']).curve_idx.nunique().reset_index()

,test_kit,target,curve_idx
0,LuNER,E gene,8480
1,LuNER,N gene,8480
2,LuNER,RnaseP,178094
3,Thermo,MS2,40444
4,Thermo,N gene,2059
5,Thermo,ORF1ab,2059
6,Thermo,S gene,2059


In [33]:
testkit_df = groundtruth.groupby(['curve_idx']).cycle_no.max().reset_index()
luner_testkit_df = testkit_df[testkit_df.cycle_no == 45]
groundtruth['test_kit'] = 'Thermo'
groundtruth.loc[groundtruth.curve_idx.isin(luner_testkit_df.curve_idx),'test_kit'] = 'LuNER'
groundtruth.groupby(['test_kit','target']).sample_id.nunique()

test_kit  target
LuNER     E gene      8496
          N gene      8496
          RnaseP    178111
Thermo    MS2        40454
          N gene      2067
          ORF1ab      2067
          S gene      2067
Name: sample_id, dtype: int64

In [24]:
print('Number of unique pcr plates:', join_df.pcr_plate.nunique())
print('Number of unique samples:', join_df.sample_id.nunique())
print('Number of unique curves:', join_df.curve_idx.nunique())
print('Number of retest samples', join_df[~join_df.retest_sample_id_1.isna()].sample_id.nunique())
print('Number of samples retest at least twice', join_df[~join_df.retest_sample_id_2.isna()].sample_id.nunique())
print('Earliest testing date:', join_df.created_date.min())
print('Last testing date:', join_df.created_date.max())

Number of unique pcr plates: 1201
Number of unique samples: 226445
Number of unique curves: 721684
Number of retest samples 5476
Number of samples retest at least twice 349
Earliest testing date: 2020-04-21 00:00:00
Last testing date: 2022-08-02 00:00:00


In [26]:
testkit_df = join_df.groupby(['curve_idx']).cycle_no.max().reset_index()
luner_testkit_df = testkit_df[testkit_df.cycle_no == 45]
join_df['test_kit'] = 'Thermo'
join_df.loc[join_df.curve_idx.isin(luner_testkit_df.curve_idx),'test_kit'] = 'LuNER'
join_df.groupby('test_kit').curve_idx.nunique()

test_kit
LuNER     551964
Thermo    169720
Name: curve_idx, dtype: int64

In [29]:
join_df.groupby('sample_type').sample_id.nunique().reset_index()

,sample_type,sample_id
0,Buffer Negative Control (Extraction),2404
1,Clinical Sample,213478
2,Human Normal Negative Control (Extraction),2404
3,Negative Control (qPCR),4215
4,Positive Control (qPCR),3944


In [16]:
join_df.head()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,...,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
0,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,1,13408.349609,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
1,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,2,13445.377930,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
2,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,3,13458.586914,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
3,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,4,13539.398438,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
4,A1,RnaseP,ATTO 647,2.503202,24.674406947379367,10000.0,3,15,5,13588.847656,...,Clinical Sample,Negative,Negative,4/27/21,Pooled Sample,NaN,NaN,db2a,Positive,24.674406947379367
